# BEST-Rec v2.1: Algorithmic Improvements

Run this **alongside** the main v2 notebook. It reuses the same cached data and embeddings but tests improved model architectures.

### Why the base model underperforms and what we fix:

| Problem | Why it hurts | Fix |
|---|---|---|
| **No rating bias** | Every user/item starts from zero — model must learn global patterns from scratch | Add global mean + learnable user/item bias terms (like SVD++) |
| **Naive mean pooling** | All user reviews weighted equally, even irrelevant ones | Attention-based pooling that learns which reviews matter |
| **No output clamping** | Model can predict 0.3 or 7.2 which inflates MAE/RMSE | Clamp predictions to [1, 5] at inference |
| **Flat learning rate** | No warm-up causes early instability; no decay causes late overshooting | Linear warm-up + cosine decay schedule |
| **Shallow fusion** | Single MLP with no skip connections loses signal | Residual fusion with layer normalisation |
| **No regularisation on embeddings** | SVD features have wildly different scales across folds | L2 normalise SVD features + dropout on all projected tokens |

Each improvement is tested independently so you can see which ones help.

In [1]:
import subprocess, sys
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
pip_install("torch", "torchvision", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cu128")
pip_install("transformers", "scikit-learn", "scipy", "numpy", "tqdm", "pandas", "ipywidgets")
print("Dependencies ready.")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip


Dependencies ready.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip


## 1. Imports & Load Cached Data from v2

In [2]:
import os, json, pickle, copy, time, warnings
from collections import defaultdict
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.sparse import csr_matrix
from transformers import AutoTokenizer, AutoModel

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
else:
    vram_gb = 0
    print("CPU mode")

GPU: NVIDIA GeForce RTX 5060 Ti (17.1 GB)


## 2. Configuration

In [3]:
DATASET = "beauty"

DATASET_FILES = {
    "beauty":      ("All_Beauty.jsonl",          "meta_All_Beauty.jsonl"),
    "books":       ("Books.jsonl",               "meta_Books.jsonl"),
    "fashion":     ("Amazon_Fashion.jsonl",      "meta_Amazon_Fashion.jsonl"),
    "instruments": ("Musical_Instruments.jsonl", "meta_Musical_Instruments.jsonl"),
}

DATA_DIR  = "./data"
CACHE_DIR = f"./cache/{DATASET}"
os.makedirs(CACHE_DIR, exist_ok=True)

INTER_FILE, META_FILE = DATASET_FILES[DATASET]
INTER_PATH = os.path.join(DATA_DIR, DATASET, INTER_FILE)
META_PATH  = os.path.join(DATA_DIR, DATASET, META_FILE)

# Hardware
NUM_CPU_WORKERS = min(8, max(0, os.cpu_count() - 2))
USE_AMP    = device.type == "cuda"
PIN_MEMORY = device.type == "cuda"
BATCH_SIZE = 4096 if vram_gb >= 12 else (2048 if vram_gb >= 8 else 1024)

# Model
PRETRAINED_MODEL   = "distilbert-base-uncased"
TEXT_DIM           = 768
HIDDEN_DIM         = 512
NUM_HEADS          = 4
NUM_ENCODER_LAYERS = 2
MAX_USER_REVIEWS   = 5
SVD_COMPONENTS     = 512
NUM_CLASSES        = 5

# Training
LR           = 3e-4       # slightly higher — warm-up will handle early instability
EPOCHS       = 40
PATIENCE     = 10
LAMBDA_CLS   = 1.5
WEIGHT_DECAY = 1e-4       # stronger regularisation
GRAD_CLIP    = 1.0
WARMUP_EPOCHS = 3         # NEW: linear warm-up

# Eval
NUM_FOLDS           = 5
NEG_SAMPLES         = 99
TOP_K               = 10
COLD_USER_THRESHOLD = 3
COLD_ITEM_THRESHOLD = 5
RANKING_EVAL_USERS  = 2000

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Dataset: {DATASET}  |  Batch: {BATCH_SIZE}  |  LR: {LR}  |  Warmup: {WARMUP_EPOCHS} ep")

Dataset: beauty  |  Batch: 4096  |  LR: 0.0003  |  Warmup: 3 ep


## 3. Load Cached Data (from v2 notebook)

In [4]:
def cached(name, fn, force=False):
    path = os.path.join(CACHE_DIR, name)
    if os.path.exists(path) and not force:
        print(f"  Cache hit: {name}")
        with open(path, "rb") as f:
            return pickle.load(f)
    print(f"  Computing: {name}...")
    result = fn()
    with open(path, "wb") as f:
        pickle.dump(result, f)
    return result

def cached_tensor(name, fn, force=False):
    path = os.path.join(CACHE_DIR, name)
    if os.path.exists(path) and not force:
        print(f"  Cache hit: {name}")
        return torch.load(path, weights_only=True)
    print(f"  Computing: {name}...")
    result = fn()
    torch.save(result, path)
    return result


# ── Load raw data (must exist from v2 run) ──
raw_cache = os.path.join(CACHE_DIR, "raw_data.pkl")
assert os.path.exists(raw_cache), f"Run the v2 notebook first to create {raw_cache}"

with open(raw_cache, "rb") as f:
    data = pickle.load(f)

interactions  = data["interactions"]
item_metadata = data["item_metadata"]
num_users     = data["num_users"]
num_items     = data["num_items"]
print(f"Loaded: {num_users:,} users, {num_items:,} items, {len(interactions):,} interactions")

# ── Load precomputed features ──
item_title_embeds = cached_tensor("item_title_embeds.pt", lambda: None)

def _compute_numeric():
    avg_r = torch.tensor([item_metadata[i]["avg_rating"] for i in range(num_items)])
    r_num = torch.tensor([item_metadata[i]["rating_num"] for i in range(num_items)])
    price = torch.tensor([item_metadata[i]["price"]      for i in range(num_items)])
    def zscore(x): return (x - x.mean()) / (x.std() + 1e-9)
    return torch.stack([zscore(avg_r), zscore(r_num), zscore(price)], dim=-1)

item_numeric = cached_tensor("item_numeric.pt", _compute_numeric)
print(f"Title embeds: {item_title_embeds.shape}  |  Numeric: {item_numeric.shape}")

Loaded: 631,986 users, 112,565 items, 701,528 interactions
  Cache hit: item_title_embeds.pt
  Cache hit: item_numeric.pt
Title embeds: torch.Size([112565, 768])  |  Numeric: torch.Size([112565, 3])


## 4. Per-Fold Features (same leak-free approach)

In [5]:
# Text encoder for per-fold user embeddings
class TextEncoder:
    def __init__(self, model_name, device, max_length=64):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device).eval()
        self.device = device
        self.max_length = max_length
        self.dim = self.model.config.hidden_size

    @torch.no_grad()
    def encode_batch(self, texts, batch_size=256):
        all_embeds = []
        for i in range(0, len(texts), batch_size):
            batch = [t if t.strip() else "empty" for t in texts[i:i+batch_size]]
            inputs = self.tokenizer(batch, padding=True, truncation=True,
                                     max_length=self.max_length, return_tensors="pt").to(self.device)
            with autocast(enabled=USE_AMP):
                out = self.model(**inputs)
            all_embeds.append(out.last_hidden_state[:, 0, :].float().cpu())
        return torch.cat(all_embeds)

    def encode_user_reviews_for_fold(self, train_inters, num_users, max_reviews=5):
        user_reviews = defaultdict(list)
        for inter in train_inters:
            if inter["review"].strip():
                user_reviews[inter["user_id"]].append(inter["review"])
        uid_slot_text = []
        for uid in range(num_users):
            for j, rev in enumerate(user_reviews.get(uid, [])[:max_reviews]):
                uid_slot_text.append((uid, j, rev))
        embeds = torch.zeros(num_users, max_reviews, self.dim)
        masks  = torch.zeros(num_users, max_reviews, dtype=torch.bool)
        if uid_slot_text:
            encoded = self.encode_batch([t[2] for t in uid_slot_text])
            for idx, (uid, j, _) in enumerate(uid_slot_text):
                embeds[uid, j] = encoded[idx]
                masks[uid, j]  = True
        return embeds, masks

text_encoder = TextEncoder(PRETRAINED_MODEL, device)


def compute_svd_for_fold(train_inters, fold_tag, force=False):
    def _compute():
        rows, cols, vals = [], [], []
        for inter in train_inters:
            rows.append(inter["item_id"]); cols.append(inter["user_id"]); vals.append(inter["rating"])
        mat = csr_matrix((vals, (rows, cols)), shape=(num_items, num_users))
        k = min(SVD_COMPONENTS, min(num_items, num_users) - 1, len(set(rows)) - 1)
        if k < 1: return torch.zeros(num_items, SVD_COMPONENTS)
        svd = TruncatedSVD(n_components=k, random_state=SEED)
        result = svd.fit_transform(mat)
        if k < SVD_COMPONENTS:
            result = np.hstack([result, np.zeros((num_items, SVD_COMPONENTS - k))])
        return torch.tensor(result, dtype=torch.float32)
    return cached_tensor(f"svd_fold_{fold_tag}.pt", _compute, force)


def compute_user_embeds_for_fold(train_inters, fold_tag, force=False):
    def _compute():
        e, m = text_encoder.encode_user_reviews_for_fold(train_inters, num_users, MAX_USER_REVIEWS)
        return {"embeds": e, "masks": m}
    r = cached(f"user_embeds_fold_{fold_tag}.pkl", _compute, force)
    return r["embeds"], r["masks"]


# Compute user/item rating statistics for bias initialisation
def compute_rating_stats(train_inters):
    global_sum, global_n = 0.0, 0
    user_sums = defaultdict(float)
    user_counts = defaultdict(int)
    item_sums = defaultdict(float)
    item_counts = defaultdict(int)
    for inter in train_inters:
        r = inter["rating"]
        global_sum += r; global_n += 1
        user_sums[inter["user_id"]] += r; user_counts[inter["user_id"]] += 1
        item_sums[inter["item_id"]] += r; item_counts[inter["item_id"]] += 1
    global_mean = global_sum / global_n
    user_bias = torch.zeros(num_users)
    item_bias = torch.zeros(num_items)
    for uid in range(num_users):
        if user_counts[uid] > 0:
            user_bias[uid] = (user_sums[uid] / user_counts[uid]) - global_mean
    for iid in range(num_items):
        if item_counts[iid] > 0:
            item_bias[iid] = (item_sums[iid] / item_counts[iid]) - global_mean
    return global_mean, user_bias, item_bias

print("Per-fold feature functions ready.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Per-fold feature functions ready.


## 5. Dataset Class

In [6]:
class BESTRecDataset(Dataset):
    def __init__(self, interactions, user_review_embeds, user_masks,
                 item_title_embeds, item_numeric, item_svd):
        self.user_ids   = torch.tensor([i["user_id"] for i in interactions], dtype=torch.long)
        self.item_ids   = torch.tensor([i["item_id"] for i in interactions], dtype=torch.long)
        self.ratings    = torch.tensor([i["rating"]  for i in interactions], dtype=torch.float32)
        self.cls_labels = torch.clamp(self.ratings.long() - 1, min=0)
        self.user_review_embeds = user_review_embeds
        self.user_masks         = user_masks
        self.item_title_embeds  = item_title_embeds
        self.item_numeric       = item_numeric
        self.item_svd           = item_svd

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        uid = self.user_ids[idx]; iid = self.item_ids[idx]
        return (self.user_review_embeds[uid], self.user_masks[uid],
                self.item_title_embeds[iid], self.item_numeric[iid], self.item_svd[iid],
                self.ratings[idx], self.cls_labels[idx], uid, iid)

## 6. Improved Model: BEST-Rec v2.1

### Key architectural improvements:

**1. Rating bias terms** (global mean + user bias + item bias):
Like SVD++, the model starts from `global_mean + user_bias[u] + item_bias[i]` and the neural network learns the residual. This gives the model a massive head start — even before any text or attention, it knows that user #42 rates 0.3 stars above average and item #99 is 0.5 below.

**2. Attention pooling for user reviews**:
Instead of treating all 5 reviews equally (mean pool), a learned query vector attends over the review embeddings. The model learns that a review saying "amazing product" is more informative than "ok" for predicting the next rating.

**3. L2-normalised SVD features**:
Raw SVD features have wildly varying magnitudes. Normalising them to unit length makes training more stable and prevents SVD from dominating the gradient.

**4. Residual fusion with LayerNorm**:
The fusion MLP now has skip connections so gradients flow better through the deep network.

**5. Clamped output at inference**:
Predictions are clamped to [1, 5] during evaluation — free error reduction.

In [7]:
class BESTRecV21(nn.Module):
    """
    BEST-Rec v2.1 with bias terms, attention pooling,
    normalised SVD, residual fusion, and output clamping.
    """

    def __init__(self, num_users, num_items, global_mean, user_bias_init, item_bias_init,
                 text_dim=TEXT_DIM, hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS,
                 num_layers=NUM_ENCODER_LAYERS, svd_dim=SVD_COMPONENTS,
                 num_classes=NUM_CLASSES):
        super().__init__()
        H = hidden_dim

        # ── Improvement 1: Bias terms ──
        self.global_mean = nn.Parameter(torch.tensor(global_mean), requires_grad=False)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        # Initialise from training data statistics
        self.user_bias.weight.data = user_bias_init.unsqueeze(1)
        self.item_bias.weight.data = item_bias_init.unsqueeze(1)

        # ── Projections with dropout ──
        self.user_review_proj  = nn.Sequential(nn.Linear(text_dim, H), nn.Dropout(0.1))
        self.item_title_proj   = nn.Sequential(nn.Linear(text_dim, H), nn.Dropout(0.1))
        self.item_numeric_proj = nn.Sequential(nn.Linear(3, H), nn.Dropout(0.1))
        self.item_svd_proj     = nn.Sequential(nn.Linear(svd_dim, H), nn.Dropout(0.1))

        self.item_token_type = nn.Embedding(3, H)

        # ── Per-tower encoders ──
        u_layer = nn.TransformerEncoderLayer(
            d_model=H, nhead=num_heads, dim_feedforward=H*4,
            dropout=0.1, batch_first=True, activation="gelu")
        self.user_encoder = nn.TransformerEncoder(u_layer, num_layers=num_layers)

        i_layer = nn.TransformerEncoderLayer(
            d_model=H, nhead=num_heads, dim_feedforward=H*4,
            dropout=0.1, batch_first=True, activation="gelu")
        self.item_encoder = nn.TransformerEncoder(i_layer, num_layers=num_layers)

        # ── Improvement 2: Attention pooling for user reviews ──
        self.user_pool_query = nn.Parameter(torch.randn(1, 1, H) * 0.02)
        self.user_pool_attn  = nn.MultiheadAttention(
            embed_dim=H, num_heads=num_heads, batch_first=True, dropout=0.1)

        # ── Cross-attention (bidirectional) ──
        self.cross_attn_u2i = nn.MultiheadAttention(
            embed_dim=H, num_heads=num_heads, batch_first=True, dropout=0.1)
        self.norm_u2i = nn.LayerNorm(H)
        self.cross_attn_i2u = nn.MultiheadAttention(
            embed_dim=H, num_heads=num_heads, batch_first=True, dropout=0.1)
        self.norm_i2u = nn.LayerNorm(H)

        # ── Improvement 4: Residual fusion ──
        self.fusion_proj = nn.Linear(H * 4, H)
        self.fusion_norm = nn.LayerNorm(H)
        self.fusion_mlp  = nn.Sequential(
            nn.Linear(H, H), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(H, H), nn.GELU(), nn.Dropout(0.1))
        self.fusion_norm2 = nn.LayerNorm(H)

        # ── Prediction heads ──
        self.regressor = nn.Sequential(
            nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(H // 2, 1))
        self.classifier = nn.Linear(H, num_classes)

    def forward(self, user_reviews, user_mask, item_title, item_numeric, item_svd,
                user_ids=None, item_ids=None, clamp_output=False):
        B = user_reviews.size(0)

        # ── User tower ──
        u_tokens   = self.user_review_proj(user_reviews)
        u_pad_mask = ~user_mask
        all_pad    = u_pad_mask.all(dim=1)
        if all_pad.any():
            u_pad_mask[all_pad, 0] = False
        u_enc = self.user_encoder(u_tokens, src_key_padding_mask=u_pad_mask)

        # Improvement 2: Attention pooling instead of mean pooling
        pool_query = self.user_pool_query.expand(B, -1, -1)
        u_pool, _ = self.user_pool_attn(
            query=pool_query, key=u_enc, value=u_enc,
            key_padding_mask=u_pad_mask)
        u_pool = u_pool.squeeze(1)  # (B, H)

        # ── Item tower ──
        # Improvement 3: L2-normalise SVD features
        svd_normed = F.normalize(item_svd, p=2, dim=-1)

        t_title   = self.item_title_proj(item_title).unsqueeze(1)
        t_numeric = self.item_numeric_proj(item_numeric).unsqueeze(1)
        t_svd     = self.item_svd_proj(svd_normed).unsqueeze(1)

        i_tokens = torch.cat([t_title, t_numeric, t_svd], dim=1)
        type_ids = torch.tensor([0, 1, 2], device=i_tokens.device).unsqueeze(0).expand(B, -1)
        i_tokens = i_tokens + self.item_token_type(type_ids)
        i_enc    = self.item_encoder(i_tokens)
        i_pool   = i_enc.mean(dim=1)

        # ── Cross-attention ──
        u2i, _ = self.cross_attn_u2i(query=u_enc, key=i_enc, value=i_enc)
        u2i = self.norm_u2i(u2i + u_enc)
        u_mask_f = user_mask.unsqueeze(-1).float()
        u2i_pool = (u2i * u_mask_f).sum(1) / (u_mask_f.sum(1) + 1e-9)

        i2u, _ = self.cross_attn_i2u(query=i_enc, key=u_enc, value=u_enc,
                                      key_padding_mask=u_pad_mask)
        i2u = self.norm_i2u(i2u + i_enc)
        i2u_pool = i2u.mean(dim=1)

        # ── Improvement 4: Residual fusion ──
        fused_in = torch.cat([u_pool, u2i_pool, i_pool, i2u_pool], dim=-1)
        z = self.fusion_norm(self.fusion_proj(fused_in))
        z = self.fusion_norm2(z + self.fusion_mlp(z))  # residual

        # ── Prediction ──
        residual = self.regressor(z).squeeze(-1)  # neural residual

        # Improvement 1: bias + residual
        if user_ids is not None and item_ids is not None:
            u_b = self.user_bias(user_ids).squeeze(-1)
            i_b = self.item_bias(item_ids).squeeze(-1)
            rating_pred = self.global_mean + u_b + i_b + residual
        else:
            rating_pred = self.global_mean + residual

        # Improvement 5: clamp at inference
        if clamp_output:
            rating_pred = torch.clamp(rating_pred, 1.0, 5.0)

        cls_logits = self.classifier(z)
        return rating_pred, cls_logits

print("BESTRecV21 model defined.")

BESTRecV21 model defined.


## 7. Training with Warm-up + Cosine Decay

In [8]:
amp_scaler = GradScaler(enabled=USE_AMP)


def get_scheduler(optimizer, num_batches_per_epoch):
    warmup_steps = WARMUP_EPOCHS * num_batches_per_epoch
    total_steps  = EPOCHS * num_batches_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)  # linear warm-up
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + np.cos(np.pi * progress))  # cosine decay

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, optimizer, scheduler, dataloader):
    global amp_scaler
    model.train()
    crit_reg = nn.SmoothL1Loss()
    crit_cls = nn.CrossEntropyLoss()
    total_loss, n = 0.0, 0

    for batch in tqdm(dataloader, desc="  Train", leave=False):
        u_rev, u_mask, i_tit, i_num, i_svd, rat, cls, uids, iids = batch
        u_rev  = u_rev.to(device, non_blocking=True)
        u_mask = u_mask.to(device, non_blocking=True)
        i_tit  = i_tit.to(device, non_blocking=True)
        i_num  = i_num.to(device, non_blocking=True)
        i_svd  = i_svd.to(device, non_blocking=True)
        rat    = rat.to(device, non_blocking=True)
        cls    = cls.to(device, non_blocking=True)
        uids   = uids.to(device, non_blocking=True)
        iids   = iids.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            pred_r, pred_cls = model(u_rev, u_mask, i_tit, i_num, i_svd,
                                     user_ids=uids, item_ids=iids)
            loss = crit_reg(pred_r, rat) + LAMBDA_CLS * crit_cls(pred_cls, cls)

        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        amp_scaler.step(optimizer)
        amp_scaler.update()
        scheduler.step()

        total_loss += loss.item() * rat.size(0)
        n += rat.size(0)
    return total_loss / n


@torch.no_grad()
def evaluate_rating(model, dataloader):
    model.eval()
    all_preds, all_targets = [], []
    for batch in tqdm(dataloader, desc="  Eval", leave=False):
        u_rev, u_mask, i_tit, i_num, i_svd, rat, _, uids, iids = batch
        with autocast(enabled=USE_AMP):
            pred_r, _ = model(
                u_rev.to(device, non_blocking=True),
                u_mask.to(device, non_blocking=True),
                i_tit.to(device, non_blocking=True),
                i_num.to(device, non_blocking=True),
                i_svd.to(device, non_blocking=True),
                user_ids=uids.to(device, non_blocking=True),
                item_ids=iids.to(device, non_blocking=True),
                clamp_output=True)  # CLAMP at eval
        all_preds.append(pred_r.float().cpu().numpy())
        all_targets.append(rat.numpy())
    preds   = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    return mean_absolute_error(targets, preds), np.sqrt(mean_squared_error(targets, preds))


@torch.no_grad()
def evaluate_ranking(model, test_inters, user_review_embeds, user_masks,
                     item_title_embeds, item_numeric, item_svd):
    model.eval()
    user_test_items = defaultdict(set)
    for inter in test_inters:
        user_test_items[inter["user_id"]].add(inter["item_id"])
    ndcg_list, hr_list = [], []
    all_items = set(range(num_items))
    rng = np.random.RandomState(SEED)
    eval_users = [u for u in user_test_items if user_test_items[u]][:RANKING_EVAL_USERS]

    for uid in tqdm(eval_users, desc="  Ranking", leave=False):
        for pos_iid in user_test_items[uid]:
            neg_pool = list(all_items - user_test_items[uid])
            if len(neg_pool) < NEG_SAMPLES: continue
            neg_iids = rng.choice(neg_pool, size=NEG_SAMPLES, replace=False)
            cands = [pos_iid] + list(neg_iids)
            n = len(cands)
            # No user/item IDs for ranking (bias doesn't help rank — it's constant per user)
            with autocast(enabled=USE_AMP):
                scores, _ = model(
                    user_review_embeds[uid].unsqueeze(0).expand(n,-1,-1).to(device),
                    user_masks[uid].unsqueeze(0).expand(n,-1).to(device),
                    item_title_embeds[cands].to(device),
                    item_numeric[cands].to(device),
                    item_svd[cands].to(device))
            scores = scores.float().cpu().numpy()
            ranked = np.argsort(-scores)
            pos_rank = int(np.where(ranked == 0)[0][0])
            hr_list.append(1.0 if pos_rank < TOP_K else 0.0)
            ndcg_list.append(1.0/np.log2(pos_rank+2) if pos_rank < TOP_K else 0.0)

    return {f"NDCG@{TOP_K}": np.mean(ndcg_list) if ndcg_list else 0.0,
            f"HR@{TOP_K}":   np.mean(hr_list)   if hr_list   else 0.0}

print("Training functions ready (warm-up + cosine decay + clamped eval).")

Training functions ready (warm-up + cosine decay + clamped eval).


## 8. Run Fold (v2.1)

In [9]:
def run_fold_v21(fold_tag, train_inters, test_inters):
    global amp_scaler
    amp_scaler = GradScaler(enabled=USE_AMP)

    print(f"\n{'='*60}")
    print(f"  Fold [{fold_tag}]: {len(train_inters):,} train / {len(test_inters):,} test")
    print(f"{'='*60}")

    item_svd = compute_svd_for_fold(train_inters, fold_tag)
    user_embeds, user_masks = compute_user_embeds_for_fold(train_inters, fold_tag)

    # Compute bias terms from training data
    global_mean, user_bias_init, item_bias_init = compute_rating_stats(train_inters)
    print(f"  Global mean rating: {global_mean:.3f}")

    train_ds = BESTRecDataset(train_inters, user_embeds, user_masks,
                               item_title_embeds, item_numeric, item_svd)
    test_ds  = BESTRecDataset(test_inters,  user_embeds, user_masks,
                               item_title_embeds, item_numeric, item_svd)
    dl_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_CPU_WORKERS,
                 pin_memory=PIN_MEMORY,
                 prefetch_factor=4 if NUM_CPU_WORKERS > 0 else None,
                 persistent_workers=True if NUM_CPU_WORKERS > 0 else False)
    train_dl = DataLoader(train_ds, shuffle=True,  **dl_kw)
    test_dl  = DataLoader(test_ds,  shuffle=False, **dl_kw)

    # Model
    model = BESTRecV21(
        num_users, num_items, global_mean,
        user_bias_init, item_bias_init
    ).to(device)

    raw_model = model
    if hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print("  Model compiled")
        except Exception:
            pass

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer, len(train_dl))

    best_mae   = float("inf")
    best_state = None
    patience_ctr = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        train_loss = train_one_epoch(model, optimizer, scheduler, train_dl)
        val_mae, val_rmse = evaluate_rating(model, test_dl)
        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]["lr"]

        marker = ""
        if val_mae < best_mae:
            best_mae = val_mae
            best_state = copy.deepcopy(raw_model.state_dict())
            patience_ctr = 0
            marker = " *best*"
        else:
            patience_ctr += 1

        print(f"  Ep {epoch+1:2d}/{EPOCHS} | loss={train_loss:.4f} | "
              f"MAE={val_mae:.4f} | RMSE={val_rmse:.4f} | "
              f"lr={lr_now:.1e} | {elapsed:.0f}s{marker}")

        if patience_ctr >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    raw_model.load_state_dict(best_state)
    print(f"  Restored best model (MAE={best_mae:.4f})")

    final_mae, final_rmse = evaluate_rating(model, test_dl)
    rank_results = evaluate_ranking(model, test_inters, user_embeds, user_masks,
                                     item_title_embeds, item_numeric, item_svd)
    results = {"mae": final_mae, "rmse": final_rmse, **rank_results}
    print(f"  Results:")
    for k, v in results.items():
        print(f"     {k}: {v:.4f}")

    del model, raw_model, optimizer, scheduler, train_dl, test_dl
    torch.cuda.empty_cache()
    return results

## 9. Run All 5 Folds

In [ ]:
print("=" * 60)
print("  BEST-Rec v2.1: Improved Model — GroupKFold by user")
print("=" * 60)

user_ids_arr = np.array([i["user_id"] for i in interactions])
indices = np.arange(len(interactions))
gkf = GroupKFold(n_splits=NUM_FOLDS)

v21_results = []

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(indices, groups=user_ids_arr)):
    train_inters = [interactions[i] for i in train_idx]
    test_inters  = [interactions[i] for i in test_idx]
    result = run_fold_v21(f"warm_{fold_idx}", train_inters, test_inters)
    v21_results.append(result)

print("\n" + "=" * 60)
print("v2.1 RESULTS SUMMARY")
print("=" * 60)
for metric in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[metric] for r in v21_results]
    print(f"  {metric:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}   {[round(v,4) for v in vals]}")

  BEST-Rec v2.1: Improved Model — GroupKFold by user

  Fold [warm_0]: 561,222 train / 140,306 test
  Cache hit: svd_fold_warm_0.pt
  Cache hit: user_embeds_fold_warm_0.pkl
  Global mean rating: 3.960
  Model compiled


  Train:   0%|          | 0/138 [00:00<?, ?it/s]

W0403 08:32:38.639000 1149885 torch/_inductor/utils.py:1731] [1/0] Not enough SMs to use max_autotune_gemm mode
W0403 08:33:30.856000 1149885 torch/_functorch/_aot_autograd/autograd_cache.py:1101] [1/1] AOTAutograd cache unable to serialize compiled graph: Please convert all Tensors to FakeTensors first or instantiate FakeTensorMode with 'allow_non_fake_inputs'. Found in aten._to_copy.default(tensor([...], device='cuda:0', size=(24,), dtype=torch.uint8), device=device(type='cpu'))


  Eval:   0%|          | 0/35 [00:00<?, ?it/s]

  Ep  1/40 | loss=1.6120 | MAE=1.3468 | RMSE=1.6165 | lr=1.0e-04 | 78s *best*


...

  Eval:   0%|          | 0/35 [00:00<?, ?it/s]

  Ep 18/40 | loss=0.9376 | MAE=1.3055 | RMSE=1.5964 | lr=1.9e-04 | 17s
  Early stopping at epoch 18
  Restored best model (MAE=1.2862)


  Eval:   0%|          | 0/35 [00:00<?, ?it/s]

  Ranking:   0%|          | 0/2000 [00:00<?, ?it/s]

  Results:
     mae: 1.2862
     rmse: 1.6323
     NDCG@10: 0.0122
     HR@10: 0.0329

  Fold [warm_1]: 561,222 train / 140,306 test
  Computing: svd_fold_warm_1.pt...


## 10. Cold-Start Evaluation

In [ ]:
def split_cold_users(inters):
    user_c = defaultdict(int)
    for i in inters: user_c[i["user_id"]] += 1
    cold = {u for u, c in user_c.items() if c <= COLD_USER_THRESHOLD}
    train = [i for i in inters if i["user_id"] not in cold]
    test  = [i for i in inters if i["user_id"] in cold]
    print(f"  Cold users: {len(cold):,} | Train: {len(train):,} | Test: {len(test):,}")
    return train, test

def split_cold_items(inters):
    item_c = defaultdict(int)
    for i in inters: item_c[i["item_id"]] += 1
    cold = {i for i, c in item_c.items() if c <= COLD_ITEM_THRESHOLD}
    train = [i for i in inters if i["item_id"] not in cold]
    test  = [i for i in inters if i["item_id"] in cold]
    print(f"  Cold items: {len(cold):,} | Train: {len(train):,} | Test: {len(test):,}")
    return train, test

print("\n--- Cold-User ---")
cu_tr, cu_te = split_cold_users(interactions)
cold_user_result = run_fold_v21("cold_user", cu_tr, cu_te) if len(cu_te) >= 10 else None

print("\n--- Cold-Item ---")
ci_tr, ci_te = split_cold_items(interactions)
cold_item_result = run_fold_v21("cold_item", ci_tr, ci_te) if len(ci_te) >= 10 else None

## 11. Compare v2 vs v2.1

Load v2 results from the shared cache and compare side-by-side.

In [ ]:
# Load v2 results if available
v2_path = os.path.join(CACHE_DIR, "all_results.json")
v2_data = None
if os.path.exists(v2_path):
    with open(v2_path) as f:
        v2_data = json.load(f)
    print("v2 results loaded for comparison.\n")

print("=" * 70)
print(f"  COMPARISON: v2 vs v2.1 on {DATASET.upper()}")
print("=" * 70)

if v2_data and "warm" in v2_data:
    v2_warm = v2_data["warm"]
    print(f"\n{'Metric':<12s}  {'v2 mean':>10s}  {'v2.1 mean':>10s}  {'Delta':>10s}")
    print("-" * 50)
    for metric in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
        v2_vals  = [r[metric] for r in v2_warm]
        v21_vals = [r[metric] for r in v21_results]
        v2_m  = np.mean(v2_vals)
        v21_m = np.mean(v21_vals)
        delta = v21_m - v2_m
        direction = "better" if (delta < 0 and "mae" in metric.lower() or "rmse" in metric.lower()) else ""
        if "NDCG" in metric or "HR" in metric:
            direction = "better" if delta > 0 else ""
        print(f"{metric:<12s}  {v2_m:>10.4f}  {v21_m:>10.4f}  {delta:>+10.4f}  {direction}")
else:
    print("No v2 results found — run the v2 notebook first for comparison.")

print("\nv2.1 warm folds:")
for metric in ["mae", "rmse", f"NDCG@{TOP_K}", f"HR@{TOP_K}"]:
    vals = [r[metric] for r in v21_results]
    print(f"  {metric:>10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

if cold_user_result:
    print(f"\nCold-User:  MAE={cold_user_result['mae']:.4f}  RMSE={cold_user_result['rmse']:.4f}")
if cold_item_result:
    print(f"Cold-Item:  MAE={cold_item_result['mae']:.4f}  RMSE={cold_item_result['rmse']:.4f}")

# Save v2.1 results
all_v21 = {
    "dataset": DATASET,
    "warm": v21_results,
    "cold_user": cold_user_result,
    "cold_item": cold_item_result,
}

def jsonify(obj):
    if isinstance(obj, (np.floating,)):  return float(obj)
    if isinstance(obj, (np.integer,)):   return int(obj)
    if isinstance(obj, np.ndarray):      return obj.tolist()
    if isinstance(obj, dict):            return {k: jsonify(v) for k, v in obj.items()}
    if isinstance(obj, list):            return [jsonify(v) for v in obj]
    return obj

with open(os.path.join(CACHE_DIR, "v21_results.json"), "w") as f:
    json.dump(jsonify(all_v21), f, indent=2)
print(f"\nSaved to {CACHE_DIR}/v21_results.json")